In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("process_log.csv")
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 32,058
Columns: 10


In [3]:
display(df.head(5))
display(df.info())
display(df.describe(include="all").T)

,timestamp,instance_id,state,file_name,n_pages,time_since_last_event,log_time_since_last_event,type,state_name,type_name
0,2014-01-01T08:21:53Z,0,7,NaN,NaN,0,0.000000,2,Notification of claim,Parking damage
1,2014-01-01T08:25:55Z,0,1,NaN,NaN,242,5.493061,2,Processing,Parking damage
2,2014-04-02T15:51:14Z,0,8,00003.pdf,6.0,7889119,15.880995,2,Damage documents,Parking damage
3,2014-04-03T15:32:06Z,0,5,NaN,NaN,85252,11.353379,2,Audit passed,Parking damage
4,2014-01-02T13:49:04Z,1,7,NaN,NaN,0,0.000000,1,Notification of claim,Rodent bites


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32058 entries, 0 to 32057
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  32058 non-null  object 
 1   instance_id                32058 non-null  int64  
 2   state                      32058 non-null  int64  
 3   file_name                  16132 non-null  object 
 4   n_pages                    16132 non-null  float64
 5   time_since_last_event      32058 non-null  int64  
 6   log_time_since_last_event  32058 non-null  float64
 7   type                       32058 non-null  int64  
 8   state_name                 32058 non-null  object 
 9   type_name                  32058 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 2.4+ MB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,32058,32051,2016-05-28T23:59:13Z,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
instance_id,32058.0,NaN,NaN,NaN,2666.759623,1458.794538,0.0,1433.0,2735.0,3917.0,5130.0
state,32058.0,NaN,NaN,NaN,5.977572,3.232951,0.0,3.0,7.0,8.0,11.0
file_name,16132,16132,32058.pdf,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_pages,16132.0,NaN,NaN,NaN,2.432556,2.171804,1.0,1.0,1.0,4.0,10.0
time_since_last_event,32058.0,NaN,NaN,NaN,194435.2794,1078208.944491,0.0,17.0,118.0,7581.0,61880003.0
log_time_since_last_event,32058.0,NaN,NaN,NaN,5.772949,4.320322,0.0,2.890372,4.779123,8.933531,17.940708
type,32058.0,NaN,NaN,NaN,1.024924,1.130655,0.0,0.0,1.0,1.0,5.0
state_name,32058,12,Damage documents,7142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_name,32058,6,Rodent bites,12837,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

display(missing.sort_values("missing_percent", ascending=False))
print("Duplicate rows:", df.duplicated().sum())

,missing_count,missing_percent
n_pages,15926,49.678707
file_name,15926,49.678707
instance_id,0,0.000000
timestamp,0,0.000000
state,0,0.000000
time_since_last_event,0,0.000000
log_time_since_last_event,0,0.000000
type,0,0.000000
state_name,0,0.000000
type_name,0,0.000000


Duplicate rows: 0


In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df = df.sort_values(
    ["instance_id", "timestamp"]
).reset_index(drop=True)

In [7]:
claims = df.groupby("instance_id").agg(
    first_event=("timestamp", "min"),
    last_event=("timestamp", "max"),
    event_count=("timestamp", "size"),
    damage_type=("type_name", "first"),
    total_document_pages=("n_pages", "sum")
)

claims["duration_days"] = (
    claims["last_event"] - claims["first_event"]
).dt.total_seconds() / 86400

claims.describe().drop("count")

,event_count,total_document_pages,duration_days
mean,6.247905,7.648022,14.060337
std,2.464194,3.163890,32.387805
min,2.000000,0.000000,0.000405
25%,4.000000,5.000000,0.167251
50%,6.000000,7.000000,4.401169
75%,8.000000,9.000000,14.338235
max,16.000000,27.000000,738.269502


In [8]:
damage_counts = claims["damage_type"].value_counts()

damage_counts.to_frame("claim_count")

,claim_count
damage_type,
Rodent bites,2098
Glass damage,1976
Parking damage,699
Animal damage,320
Hit & run,22
Scorch damage,16


In [12]:
df["state_name"].value_counts().to_frame("event_count")

pd.crosstab(df["type_name"], df["state_name"])

state_name,Audit passed,Correspondence,Cover note,Damage documents,Document request,Further inquiry,Internal,Notification of claim,Notification of payment,Other,Processing,Rejection
type_name,,,,,,,,,,,,
Animal damage,248,93,40,502,244,193,85,358,233,20,320,28
Glass damage,1220,398,317,2691,707,391,226,2176,1622,74,1976,86
Hit & run,18,13,4,38,14,11,4,23,17,1,22,4
Parking damage,466,206,83,910,503,318,132,797,420,74,700,84
Rodent bites,1330,338,328,2976,897,502,282,2331,1535,107,2098,113
Scorch damage,8,5,1,25,12,6,7,17,10,1,16,3


In [18]:
(
    df.assign(hours_since_last_event=df["time_since_last_event"] / 3600)
    .groupby("state_name")["hours_since_last_event"]
    .agg(["mean", "median"])
    .rename(columns={
        "mean": "mean_hours_since_last_event",
        "median": "median_hours_since_last_event"
    })
.sort_values(by="median_hours_since_last_event", ascending=False)    
.round(2)
)

,mean_hours_since_last_event,median_hours_since_last_event
state_name,,
Other,371.62,43.13
Correspondence,136.94,37.05
Cover note,152.14,17.89
Audit passed,16.42,6.43
Further inquiry,155.51,3.87
Internal,42.21,0.10
Notification of payment,4.90,0.04
Rejection,14.59,0.03
Damage documents,139.65,0.03


In [19]:
document_events = df[df["n_pages"].notna()].copy()

document_events["n_pages"].describe()

(document_events.groupby("type_name")["n_pages"]
    .agg(["count", "mean", "median", "sum"]))

,count,mean,median,sum
type_name,,,,
Animal damage,1199,2.791493,1.0,3347.0
Glass damage,6039,2.422918,1.0,14632.0
Hit & run,92,2.760870,1.0,254.0
Parking damage,2306,2.695143,1.0,6215.0
Rodent bites,6437,2.278391,1.0,14666.0
Scorch damage,59,2.169492,1.0,128.0


In [ ]:
(
    claims.groupby("damage_type")["duration_days"]
    .agg(["count", "mean", "median", "max"])
    .sort_values("median", ascending=False)
)

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 5))
sns.countplot(
    data=claims.reset_index(),
    y="damage_type",
    order=damage_counts.index
)
plt.title("Claims by Damage Type")
plt.xlabel("Number of Claims")
plt.ylabel("Damage Type")
plt.show()

plt.figure(figsize=(10, 5))
sns.countplot(
    data=df,
    y="state_name",
    order=df["state_name"].value_counts().index
)
plt.title("Claims Process Event Frequencies")
plt.xlabel("Number of Events")
plt.ylabel("Process Stage")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=claims.reset_index(),
    x="damage_type",
    y="duration_days"
)
plt.title("Claim Duration by Damage Type")
plt.xlabel("Damage Type")
plt.ylabel("Duration (Days)")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=document_events,
    x="type_name",
    y="n_pages"
)
plt.title("Document Pages by Damage Type")
plt.xlabel("Damage Type")
plt.ylabel("Pages per Document Event")
plt.xticks(rotation=30)
plt.show()

In [ ]:
claims.to_csv("claim_summary.csv")
df.to_csv("cleaned_process_log.csv", index=False)